In [ ]:
# install python packages
%pip install "dlt[deltalake,duckdb,parquet]" streamlit

In [48]:
from collections.abc import Generator
from typing import Any

from jsonpointer import set_pointer

import dlt
from dlt.common.typing import TDataItem
from dlt.destinations import filesystem as fs_out
from dlt.extract.resource import DltResource
from dlt.extract.source import DltSource
from dlt.sources.helpers import requests
from dlt.sources.helpers.rest_client.paginators import (
    JSONResponseCursorPaginator,
)
from dlt.sources.rest_api import rest_api_source
from dlt.sources.rest_api.typing import EndpointResource


In [9]:
NMDC_API_URL = "https://api.microbiomedata.org/"

ID_FIELDS = [
    "alternative_identifiers",
    "emsl_biosample_identifiers",
    "gold_biosample_identifiers",
    "igsn_biosample_identifiers",
    "img_identifiers",
    "insdc_biosample_identifiers",
    "neon_biosample_identifiers",
]

ENV_FIELDS = [
    "env_broad_scale",
    "env_local_scale",
    "env_medium",
    "env_package",
]

COORD_FIELDS = [
    "alt",  # altitude
    "depth",
    "elev", # elevation
    "geo_loc_name",
    "lat_lon",
]

DATE_FIELDS = ["add_date", "collection_date", "mod_date"]

CORE_FIELDS = [
    "associated_studies",
    "description",
    "id",
    "name",
    "project_id",
    "samp_name",
    "type",
]

BIOSAMPLE_FIELDS = ID_FIELDS + ENV_FIELDS + COORD_FIELDS + DATE_FIELDS + CORE_FIELDS


In [10]:
def fields_param(field_list: list[str]) -> dict[str, str]:
    """Generate query parameters to specify the fields to be included in the response.

    :param field_list: list of fields to include
    :type field_list: list[str]
    :return: field query param
    :rtype: str
    """
    return {"fields": ",".join(field_list)}

def filter_param(filter_list: list[tuple[str, str|None, Any]]) -> dict[str, str]:
    """Generate query parameters to filter the results returned.

    e.g. [
        ("lat_lon.latitude", ">", 35.0),         # latitude > 35.0
        ("ecosystem_category", None, "Plants"),  # ecosystem_category is 'Plants'
        ("funding_sources.search", None, "NSF")  # funding_sources contains "NSF"
    ]

    Valid comparators (per `nmdc-runtime.api.endpoints.util.py`): "<", ">", ">=", "<="
    Omit the comparator (i.e. use None) for an exact match

    To search a field, `.search` should be appended to the field name, e.g. samp_name.search

    :param filters: tuple of field name, comparator or None, and value
    :type filters: list[tuple[str, str, str]]
    :return: filter query param
    :rtype: dict[str, str]
    """
    return {"filter": ",".join(f"{fltr[0]}:{fltr[1] or ''}{fltr[2]}" for fltr in filter_list)}


In [ ]:
# endpoint configs
VALID_ENDPOINTS = ["biosamples", "studies"]
PER_PAGE = 500

def batch_endpoint_cfg(endpoint_name: str = "", filters: dict[str, Any] | None = None, fields: dict[str, Any] | None = None) -> dict[str, Any]:
    """Generate the config for a batch endpoint.

    :param endpoint_name: path to the endpoint, minus trailing slash
    :type endpoint_name: str
    :param filters: filter query param
    :type: dict[str, str]
    :param fields: fields query param
    :type: dict[str, str]
    :return: endpoint config
    :rtype: dict[str, Any]
    """
    if not endpoint_name or endpoint_name not in VALID_ENDPOINTS:
        err_msg = f"Invalid endpoint name: {endpoint_name}"
        raise ValueError(err_msg)

    params = {
        "per_page": PER_PAGE,
        "cursor": "*",
    }

    for f in [filters, fields]:
        if f:
            if not isinstance(f, dict):
                err_msg = f"invalid format for {f}"
                raise TypeError(err_msg)
            params.update(f)

    return {
        "name": endpoint_name,
        "endpoint": {
            "path": f"{endpoint_name}/",
            "data_selector": "results",
            "params": params,
            "paginator": JSONResponseCursorPaginator(
                cursor_param="cursor", cursor_path="meta.next_cursor"
            ),
        },
    }

biosample_filters = [
    ("lat_lon.latitude", ">", 35.0),  # latitude > 35.0
    ("ecosystem_category", None, "Plants"),  # ecosystem_category is 'Plants'
    ("funding_sources.search", None, "NSF"),  # funding_sources contains "NSF"
]


biosample_endpoint = batch_endpoint_cfg("biosamples", filter_param(biosample_filters), fields_param(BIOSAMPLE_FIELDS))

assert biosample_endpoint["name"] == "biosamples"
assert biosample_endpoint["endpoint"]["path"] == "biosamples/"
assert biosample_endpoint["endpoint"]["params"] == {"per_page": PER_PAGE, "cursor": "*", "fields": ",".join(BIOSAMPLE_FIELDS), "filter": "lat_lon.latitude:>35.0,ecosystem_category:Plants,funding_sources.search:NSF"}
assert isinstance(biosample_endpoint["endpoint"]["paginator"], JSONResponseCursorPaginator)


def single_endpoint_cfg(endpoint_name: str) -> dict[str, Any]:
    """Generate the config for an endpoint returning data about one entity.

    :param endpoint_name: path to the endpoint, minus trailing slash
    :type endpoint_name: str
    :return: endpoint config
    :rtype: dict[str, Any]
    """
    return {
        "name": f"{endpoint_name}_single",
        "endpoint": {
            "path": f"{endpoint_name}/" + "{resources.entity.id}",
        }
    }

study_endpt = single_endpoint_cfg("studies")
assert study_endpt["name"] == "studies_single"
assert study_endpt["endpoint"]["path"] == "studies/{resources.entity.id}"


In [43]:

def nmdc_endpoint_src(config: dict[str, Any]) -> DltSource:
    """NMDC openapi endpoints.

    :yield: REST API generator
    :rtype: Generator[TDataItem, Any, None]
    """
    return rest_api_source(
        {
            "client": {
                "base_url": NMDC_API_URL,
            },
            "resource_defaults": {"primary_key": "id"},
            "resources": [EndpointResource(**config)],
        }
    )

def nmdc_endpoint(config: dict[str, Any]) -> Generator[TDataItem, Any, None]:
    """NMDC openapi endpoints.

    :yield: REST API generator
    :rtype: Generator[TDataItem, Any, None]
    """
    yield from nmdc_endpoint_src(config)


In [ ]:
# Retrieve data from the studies endpoint and save it to duckdb
pipeline = dlt.pipeline(pipeline_name="pipe_studies", destination="duckdb", dataset_name="studies_data")
studies_cfg = batch_endpoint_cfg("studies")
studies_rsrc = nmdc_endpoint(studies_cfg)

load_info = pipeline.run(studies_rsrc)
print(load_info)

In [ ]:
# view loaded data in streamlit app
!dlt pipeline pipe_studies show

In [ ]:
# Load biosample data into local file system, saving the data as JSONL.
# By default dlt gzip-compresses JSONL output but leaves the file extension as `.jsonl`.
# To fix this, change the `layout` parameter to include a `gz` suffix.
pipeline = dlt.pipeline(
    pipeline_name="pipe_samples",
    destination=fs_out(
        bucket_url="data_dump/nmdc", layout="{table_name}/{load_id}.{file_id}.{ext}.gz"
    ),
    dataset_name="samples_data",
    # record the import and export schemas in the schema folder
    import_schema_path="schema/nmdc/biosample/import",
    export_schema_path="schema/nmdc/biosample/export"
)
samples_cfg = batch_endpoint_cfg("biosamples")
samples_gen = nmdc_endpoint(samples_cfg)

load_info = pipeline.run(samples_gen)
print(load_info)

2025-03-23 15:34:06,166|[INFO]|63904|8367032384|dlt|client.py|_send_request:124|Making GET request to https://api.microbiomedata.org/biosamples/ with params={'per_page': 500, 'cursor': '*'}, json=None
2025-03-23 15:34:09,310|[INFO]|63904|8367032384|dlt|client.py|extract_response:253|Extracted data of type list from path results with length 500
2025-03-23 15:34:09,320|[INFO]|63904|8367032384|dlt|client.py|_send_request:124|Making GET request to https://api.microbiomedata.org/biosamples/ with params={'per_page': 500, 'cursor': 'nmdc:sys0f8y9t135'}, json=None
2025-03-23 15:34:12,160|[INFO]|63904|8367032384|dlt|client.py|extract_response:253|Extracted data of type list from path results with length 500
2025-03-23 15:34:12,169|[INFO]|63904|8367032384|dlt|client.py|_send_request:124|Making GET request to https://api.microbiomedata.org/biosamples/ with params={'per_page': 500, 'cursor': 'nmdc:sys0hghqbd83'}, json=None
2025-03-23 15:34:14,956|[INFO]|63904|8367032384|dlt|client.py|extract_respo

Pipeline pipe_samples load step completed in 0.08 seconds
1 load package(s) were loaded to destination filesystem and into dataset samples_data
The filesystem destination used file:///Users/gwg/code/kbase/credit-engine/dlt_stuff/data_dump/nmdc location to store data
Load package 1742769246.162092 is LOADED and contains no failed jobs


In [ ]:
# Save the data as JSONL, but keep some data structures as JSON.
# dlt's normaliser makes a new table when it encounters a list.
# Using the table names from the previous query, we can see which fields
# contain lists (this info could also come from the NMDC schema, but this is
# simpler).
list_fields = [
    "agrochem_addition",
    "alternative_identifiers",
    "analysis_type",
    "associated_studies",
    "biosample_categories",
    "chem_administration",
    "emsl_biosample_identifiers",
    "fertilizer_regm",
    "gold_biosample_identifiers",
    "host_diet",
    "igsn_biosample_identifiers",
    "img_identifiers",
    "insdc_biosample_identifiers",
    "misc_param",
    "perturbation",
    "sample_link",
    "water_content",
    "watering_regm",
]
pipeline = dlt.pipeline(
    pipeline_name="pipe_flat_samples",
    destination=fs_out(
        bucket_url="data_dump/nmdc", layout="{table_name}/{load_id}.{file_id}.{ext}.gz"
    ),
    # give this dataset a new name
    dataset_name="flat_samples_data",
)
samples_cfg = batch_endpoint_cfg("biosamples")

# instead of using the generator produced by the `nmdc_endpoint` function,
# edit the DltResource object and use the `apply_hints` function to alter
# the export schema.
nmdc_src: DltSource = nmdc_endpoint_src(samples_cfg)
flat_samples_gen: DltResource = nmdc_src.resources["biosamples"]

# for each of the list fields, export the data as JSON (i.e. as-is) instead of flattening it.
flat_samples_gen.apply_hints(columns={field: {"data_type": "json"} for field in list_fields})

load_info = pipeline.run(flat_samples_gen)
print(load_info)

In [ ]:
# reformat the data generated by the pipeline
@dlt.transformer
def sample_transform(item: TDataItem, study_set: set[str]) -> TDataItem:
    """Transform sample data to fit the BER Common Data Model.

    :param item: biosample data structure
    :type item: TDataItem
    :param study_set: accumulator for study IDs
    :type study_set: set[str]
    :yield: transformed biosample data structure
    :rtype: Iterator[TDataItem]
    """
    # create a new object by copying over all the CORE_FIELDS
    cdm_object = {key: item[key] for key in item if key in CORE_FIELDS}

    # add any associated studies to `study_set`
    if item.get("associated_studies"):
        study_set.update(item.get("associated_studies"))

    cdm_object["alt_ids"] = []
    cdm_object["env"] = {}

    if item.get("samp_name"):
        cdm_object["alt_names"] = [item["samp_name"]]
    else:
        cdm_object["alt_names"] = []

    # keys to remap -- either move and/or rename
    remap = [
        ("lat_lon", "/coordinates"),
        ("alt", "/coordinates/altitude"),
        ("elev", "/coordinates/elevation"),
        ("depth", "/coordinates/depth"),
        ("geo_loc_name", "/coordinates/geo_loc_name"),
        # env fields go under an `env` key
        *[(field_name, f"/env/{field_name}") for field_name in ENV_FIELDS]
    ]
    # copy over the fields to be remapped (if they exist)
    for key, mapped in remap:
        if item.get(key):
            # use jsonpointer's `set_pointer` function to move these keys
            set_pointer(cdm_object, mapped, item.get(key))

    # copy all identifiers into the alt_ids field
    for key in item:
        if key.endswith("identifiers") and item.get(key):
            cdm_object["alt_ids"].extend(item.get(key, []))

    yield cdm_object

In [ ]:
# collect biosamples, reformatting the data as we go.
pipeline = dlt.pipeline(
    pipeline_name="pipe_samples",
    destination=fs_out(bucket_url="data_dump/nmdc_delta"),
    dataset_name="samples_data",
)
samples_cfg = batch_endpoint_cfg("biosamples")
samples_gen = nmdc_endpoint(samples_cfg)

# collect the sample IDs mentioned in a set
study_set = set()
load_info = pipeline.run(
    samples_gen | sample_transform(study_set).with_name("cdm_samples"),
    table_format="delta",
)
print(study_set)

In [ ]:
# use the dlt `requests` wrapper, which has some nice tweaks to the standard `requests` package,
# including better resilience to network faults and automatic retries
@dlt.resource
def study_fetcher(studies: set[str]) -> Generator[TDataItem, Any, None]:
    """Fetch study data from the NMDC API.

    :param studies: set of study IDs to fetch
    :type studies: set[str]
    :yield: study data
    :rtype: Generator[TDataItem, Any, None]
    """
    for item in studies:
        response = requests.get(
            f"{NMDC_API_URL}/studies/{item}"
        )
        yield response.json()

@dlt.transformer
def study_transformer(item: TDataItem):
    # could implement some transformations here
    print(item)
    yield item

# now, retrieve all these studies from the studies endpoint
study_pipeline = dlt.pipeline(
    pipeline_name="piped_studies",
    destination=fs_out(bucket_url="data_dump/nmdc_delta"),
    dataset_name="studies"
)
# run the study_set gathered in the previous cell through the pipeline to fetch the related studies
load_info = study_pipeline.run(
    study_fetcher(study_set) | study_transformer, table_format="delta"
)
load_info